# 02 - Data exploration (EDA)

In [1]:
import pandas as pd

# === CONFIG ===
INPUT_CSV = 'AB_NYC_2019_cleaned.csv'

PRICE_PERCENTILES = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
TOP_N_NEIGHBOURHOODS = 10
TOP_N_HOSTS = 10

MODELLING_COLS = [
    'log_price', 'room_type_code', 'neighbourhood_group_code',
    'latitude', 'longitude', 'minimum_nights',
    'number_of_reviews', 'reviews_per_month',
    'availability_365', 'calculated_host_listings_count'
]
# === END CONFIG ===

In [2]:
def log_table(title, df, decimals=None, index=True):
    print()
    print('--- ' + title + ' ---')
    if decimals is not None:
        print(df.round(decimals).to_string(index=index))
    else:
        print(df.to_string(index=index))

In [3]:
df = pd.read_csv(INPUT_CSV)
print(f'Rows: {len(df)}')
print(f'Columns: {df.columns.tolist()}')

Rows: 48847
Columns: ['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'log_price', 'room_type_code', 'neighbourhood_group_code']


In [4]:
log_table('describe (numeric)', df.describe())
log_table('price percentiles', df['price'].describe(percentiles=PRICE_PERCENTILES))
log_table('log_price', df['log_price'].describe())


--- describe (numeric) ---
                 id       host_id      latitude     longitude         price  minimum_nights  number_of_reviews  reviews_per_month  calculated_host_listings_count  availability_365     log_price  room_type_code  neighbourhood_group_code
count  4.884700e+04  4.884700e+04  48847.000000  48847.000000  48847.000000    48847.000000       48847.000000       48847.000000                    48847.000000      48847.000000  48847.000000    48847.000000              48847.000000
mean   1.902300e+07  6.763372e+07     40.728945    -73.952176    152.774705        6.117817          23.270621           1.091014                        7.149016        112.799599      4.738025        1.503879                  1.740353
std    1.098410e+07  7.862959e+07      0.054529      0.046161    240.248498        9.245071          44.550647           1.597200                       32.968270        131.615947      0.691806        0.545310                  0.805870
min    2.539000e+03  2.43800

In [5]:
log_table('room_type counts', df['room_type'].value_counts())
log_table('room_type shares', df['room_type'].value_counts(normalize=True), decimals=3)
log_table('neighbourhood_group counts', df['neighbourhood_group'].value_counts())
log_table('neighbourhood_group shares', df['neighbourhood_group'].value_counts(normalize=True), decimals=3)
print(f'\nUnique sub-neighbourhoods: {df["neighbourhood"].nunique()}')
log_table(f'top {TOP_N_NEIGHBOURHOODS} sub-neighbourhoods by listings',
          df['neighbourhood'].value_counts().head(TOP_N_NEIGHBOURHOODS))


--- room_type counts ---
room_type
Entire home/apt    25391
Private room       22299
Shared room         1157

--- room_type shares ---
room_type
Entire home/apt    0.520
Private room       0.457
Shared room        0.024

--- neighbourhood_group counts ---
neighbourhood_group
Manhattan        21642
Brooklyn         20080
Queens            5664
Bronx             1088
Staten Island      373

--- neighbourhood_group shares ---
neighbourhood_group
Manhattan        0.443
Brooklyn         0.411
Queens           0.116
Bronx            0.022
Staten Island    0.008

Unique sub-neighbourhoods: 221

--- top 10 sub-neighbourhoods by listings ---
neighbourhood
Williamsburg          3916
Bedford-Stuyvesant    3709
Harlem                2655
Bushwick              2459
Upper West Side       1969
Hell's Kitchen        1954
East Village          1852
Upper East Side       1797
Crown Heights         1563
Midtown               1545


In [6]:
log_table('median price by neighbourhood_group x room_type',
          df.pivot_table(index='neighbourhood_group', columns='room_type', values='price', aggfunc='median'),
          decimals=0)
log_table('mean log_price by neighbourhood_group x room_type',
          df.pivot_table(index='neighbourhood_group', columns='room_type', values='log_price', aggfunc='mean'),
          decimals=3)


--- median price by neighbourhood_group x room_type ---
room_type            Entire home/apt  Private room  Shared room
neighbourhood_group                                            
Bronx                          100.0          54.0         40.0
Brooklyn                       145.0          65.0         36.0
Manhattan                      191.0          90.0         69.0
Queens                         120.0          60.0         37.0
Staten Island                  100.0          50.0         30.0

--- mean log_price by neighbourhood_group x room_type ---
room_type            Entire home/apt  Private room  Shared room
neighbourhood_group                                            
Bronx                          4.705         4.047        3.763
Brooklyn                       5.014         4.202        3.746
Manhattan                      5.310         4.552        4.274
Queens                         4.845         4.135        3.793
Staten Island                  4.793         4.036  

In [7]:
log_table('availability_365', df['availability_365'].describe())
print(f'Share with availability_365 == 0: {(df["availability_365"]==0).mean():.3f}')
log_table('reviews_per_month', df['reviews_per_month'].describe())
print(f'Share with reviews_per_month == 0: {(df["reviews_per_month"]==0).mean():.3f}')


--- availability_365 ---
count    48847.000000
mean       112.799599
std        131.615947
min          0.000000
25%          0.000000
50%         45.000000
75%        227.000000
max        365.000000
Share with availability_365 == 0: 0.358

--- reviews_per_month ---
count    48847.000000
mean         1.091014
std          1.597200
min          0.000000
25%          0.040000
50%          0.370000
75%          1.580000
max         58.500000
Share with reviews_per_month == 0: 0.205


In [8]:
log_table('correlation matrix on modelling features', df[MODELLING_COLS].corr(), decimals=3)


--- correlation matrix on modelling features ---
                                log_price  room_type_code  neighbourhood_group_code  latitude  longitude  minimum_nights  number_of_reviews  reviews_per_month  availability_365  calculated_host_listings_count
log_price                           1.000          -0.612                    -0.340     0.079     -0.326           0.048             -0.043             -0.061             0.099                           0.133
room_type_code                     -0.612           1.000                     0.157     0.006      0.184          -0.131              0.003              0.027             0.023                          -0.106
neighbourhood_group_code           -0.340           0.157                     1.000    -0.298      0.501          -0.118              0.051              0.104             0.079                          -0.120
latitude                            0.079           0.006                    -0.298     1.000      0.085          

In [9]:
log_table('calculated_host_listings_count', df['calculated_host_listings_count'].describe())
log_table(f'top {TOP_N_HOSTS} host_id by listing count',
          df.groupby('host_id').size().sort_values(ascending=False).head(TOP_N_HOSTS))


--- calculated_host_listings_count ---
count    48847.000000
mean         7.149016
std         32.968270
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        327.000000

--- top 10 host_id by listing count ---
host_id
219517861    327
107434423    232
30283594     121
137358866    103
16098958      96
12243051      96
61391963      91
22541573      87
200380610     65
1475015       52
